# 05 - Validation & Fortnightly Bulletin

Compares satellite-derived **sown area** and **yield** against Ministry ground truth, computes the **evaluation matrix** (R2, RMSE, bias), renders correlation scatter plots, and generates a fortnightly crop-condition bulletin (markdown).

In [ ]:
import sys, os
sys.path.append('..')
import pandas as pd
from src import yield_model, visualization

gt = pd.read_csv('../data/sample/ministry_ground_truth.csv')

# Satellite estimates from notebooks 02/04 (fallback demo values if not run yet)
if os.path.exists('../outputs/sown_area_estimates.csv'):
    sat_area = pd.read_csv('../outputs/sown_area_estimates.csv')
else:
    sat_area = gt[['state']].copy()
    sat_area['area_lakh_ha'] = gt.ministry_sown_area_lakh_ha * 1.02  # ~330.8 vs 324.4 scenario
comp = gt.merge(sat_area, on='state')
print('Satellite total: %.1f lakh ha | Ministry total: %.1f lakh ha'
      % (comp.area_lakh_ha.sum(), comp.ministry_sown_area_lakh_ha.sum()))
comp

In [ ]:
area_metrics = yield_model.evaluate(comp.ministry_sown_area_lakh_ha, comp.area_lakh_ha)
visualization.correlation_scatter(comp, 'ministry_sown_area_lakh_ha', 'area_lakh_ha',
                                  metrics=area_metrics, title='Sown Area: Satellite vs Ministry')
area_metrics

In [ ]:
# Yield comparison (state level)
if os.path.exists('../outputs/state_yield_forecast.csv'):
    sat_yield = pd.read_csv('../outputs/state_yield_forecast.csv')
else:
    sat_yield = gt[['state']].copy()
    sat_yield['state_yield_t_ha'] = gt.ministry_yield_t_ha * 0.98
ycomp = gt.merge(sat_yield, on='state')
yield_metrics = yield_model.evaluate(ycomp.ministry_yield_t_ha, ycomp.state_yield_t_ha)
visualization.correlation_scatter(ycomp, 'ministry_yield_t_ha', 'state_yield_t_ha',
                                  metrics=yield_metrics, title='Yield: Satellite vs Ministry')
yield_metrics

In [ ]:
# Evaluation matrix - the continuous-improvement record
ev = pd.DataFrame([
    {'variable': 'sown_area_lakh_ha', **area_metrics},
    {'variable': 'yield_t_ha', **yield_metrics},
])
ev.to_csv('../outputs/evaluation_matrix.csv', index=False)
ev

## Fortnightly bulletin generation

In [ ]:
from datetime import date
vhi_path = '../outputs/vhi_fortnightly.csv'
vhi_block = ''
if os.path.exists(vhi_path):
    v = pd.read_csv(vhi_path)
    latest = v[v.date == v.date.max()].sort_values('VHI')
    vhi_block = latest.to_markdown(index=False)

bulletin = f"""# Wheat Crop Condition Bulletin - {date.today():%d %b %Y}

## Sown area (lakh ha)
{comp[['state', 'area_lakh_ha', 'ministry_sown_area_lakh_ha']].to_markdown(index=False)}

**Satellite total:** {comp.area_lakh_ha.sum():.1f} lakh ha | **Ministry:** {comp.ministry_sown_area_lakh_ha.sum():.1f} lakh ha

## Crop health (latest fortnight VHI)
{vhi_block or '_Run notebook 03 to populate VHI table._'}

## Evaluation matrix
{ev.to_markdown(index=False)}
"""
with open('../outputs/bulletin.md', 'w') as f:
    f.write(bulletin)
print(bulletin[:600])